# Machine Learning - Predicting Success From Pre-Release Data

The target variable is 'success_combined', a weighted score of 60% sales, 15% metascore and 25% userscore, binarised at the median.

**Note:** an earlier version of this notebook included 'critic_score' as a feature, but since the success label partly depends on review scores, this created a circular relationship. The current version uses only pre-release features since we want to know whether the game will succeed before the release. To compensate for the loss of this strong predictor, feature engineering is applied to extract additional signal from the available pre-release data. Only the final version(this) of the notebook is pushed to github.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import ( accuracy_score, classification_report, roc_auc_score, precision_score, recall_score, f1_score)
from sklearn.preprocessing import StandardScaler
from datetime import datetime

%matplotlib inline

In [3]:
df = pd.read_csv("C:/Users/George/Desktop/data/combined_label.csv")
print(f"Dataset shape: {df.shape}")

Dataset shape: (8624, 22)


## 1. Feature Engineering

The original four pre-release features(genre, console, publisher, release year) are weak predictors on their own. Several derived features are added to extract more signal from the same underlying data.

**Engineered Features:**
- 'publisher_game_count' - How many games each publisher has produced(measure of size and experience)
- 'publisher_success_rate' - Historical success rate by genre(training only)
- 'genre_success_rate' - Historical success rate by genre(training only)
- 'console_success_rate' - Historical success rate by platform(training only)
- 'game_age' - Years since release(captures how older vs newer titles perform)

**Important:** all success-rate featues are computed from the training data only, then mapped onto the test data. this prevents the model from indirectly seeing test labels through aggregated statistics.

In [5]:
pub_count = df.groupby('publisher').size()
df['publisher_game_count'] = df['publisher'].map(pub_count)

current_year = datetime.now().year
df['game_age'] = current_year - df['release_year']

print(df[['publisher', 'publisher_game_count', 'game_age']].head())

        publisher  publisher_game_count  game_age
0  Rockstar Games                    73    2016.6
1  Rockstar Games                    73    2016.3
2  Rockstar Games                    73    2016.4
3  Rockstar Games                    73    2018.7
4      Activision                   523    2017.3
